# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

md = dataset.metadata
print(f"{md.name}: {md.description}\n\nPublished: {md.datePublished}\nVersion: {md.version}\nIdentifier: {md.identifier}\nLicense: {md.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In Croissant, data is structured in record sets, with each record set containing fields (which map to columns, i.e., features). Let's list all record sets and their fields by `@id`.

In [ ]:
# List all record sets and their fields
print('Record Sets:')
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    print(f"  Fields:")
    for field in rs.get('fields', []):
        print(f"    - {field['@id']} (name: {field.get('name', '')})")
    print('---')
if not dataset.metadata.record_sets:
    print('No record sets found in metadata. Attempting to list records or explore the dataset schema for possible record set(s)...')

# If record_sets are empty (as this dataset defines them in distributions/files), try listing records.
print("\nAttempting to list records found in dataset:")
try:
    record_sets = dataset.record_sets()
    print('Record sets discovered by croissant implementation:')
    for rs in record_sets:
        print(f"- {rs}")
except Exception as e:
    print(f"Error: {e}")

# Attempting to print the top-level sources/files if available
print("\nDistributions/Files:")
for dist in getattr(md, 'distribution', []):
    print(f"- @id: {dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found in the previous step.

We'll extract a primary record set (table), referencing it by its `@id`.

In [ ]:
# List available record set IDs
available_record_sets = list(dataset.record_sets())
print('Detected record sets:', available_record_sets)

# Pick a record set for analysis (use the first one for demonstration if present)
if available_record_sets:
    main_record_set_id = available_record_sets[0]
    print(f"Using record set: {main_record_set_id}")
else:
    raise RuntimeError('No record sets available!')

# Load records as DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print(f'Fields (columns) for record set {main_record_set_id}:')
print(df.columns.tolist())

# Display first 5 rows
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's:
- Select a numeric field from the table.
- Filter the DataFrame for records where this field has value above a threshold (e.g., 10).
- Normalize the numeric field values (z-score normalization).
- Optionally, group by a categorical field if present.

In [ ]:
# Automatically detect numeric fields (float or int columns)
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_fields:
    print('No numeric fields found for EDA!')
else:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")

    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (e.g., 'description' or next string column after numeric)
    cat_fields = df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for c in cat_fields:
        if c != numeric_field:
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print('No categorical field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field, and (optionally) compare groups by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print('No numeric field available for visualization!')
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we discovered a grouping categorical field, show its means
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        group_means.plot(kind='bar')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset packaged as Croissant, using the `mlcroissant` library. 

- We examined available record sets and their fields by `@id`.
- We extracted a primary table and demonstrated basic EDA, including filtering, normalization, and optionally grouping.
- We visualized numeric field distributions and group averages.

This workflow enables programmatic, FAIR-structured data exploration for downstream applications in biomedical informatics and proteomics.